# Notes on different MIMIC-IV preprocessing approaches

With PyHealth we are making a subset of the whole patients (at least, it seems to be), here instead we prefer to focus more on the data, trying to understand all the info contained.

## Description of text-summary data preprocessing in preprocessiong_no_pyhealth.ipynb

This Python script processes clinical data from the MIMIC-IV database to generate structured clinical histories for patients. The data includes patient demographics, hospital admissions, ICU stays, diagnoses, prescriptions, and procedures. The script performs the following key steps:

1. **Loading Data:** Load various datasets representing different aspects of patient information, such as demographics, hospital admissions, ICU stays, prescriptions, and medical procedures.

2. **Mapping ICD Codes to Descriptions:** Merge diagnoses and procedures with their respective ICD code descriptions to provide a more comprehensive understanding of patient records.

3. **Computing Age at Events:** Calculate the age of patients at the time of different medical events such as hospital admissions, ICU stays, and prescriptions. This step includes calculating the age at death for deceased patients.

4. **Generating Structured Clinical History:** For each patient, compile a structured clinical history that includes details about admissions, diagnoses, procedures, prescribed medications, and ICU stays. The script also determines the patient's age and records any deceased status.

5. **Generating and Saving Summaries:** Create clinical history summaries for each patient and save them to a file for further analysis or reporting.

### Detailed Documentation

#### Step 1: Load Data

- Import necessary libraries and load datasets using `pandas.read_csv` to read from CSV files.
- The datasets include:
  - `patients`: Demographic information.
  - `admissions`: Hospital admission records.
  - `icustays`: ICU stay records.
  - `diagnoses`: ICD codes for patient diagnoses.
  - `prescriptions`: Medications administered to patients.
  - `procedures`: ICD codes for medical procedures.
  - `icd_diagnoses`, `icd_procedures`: Descriptions of ICD codes for diagnoses and procedures respectively.
- Convert ICU stay timestamps ('intime' and 'outtime') to `datetime` format for further processing.

#### Step 2: Map ICD Codes to Descriptions

- Merge diagnosis and procedure data with their respective ICD descriptions using a left join (`merge`) on `icd_code` and `icd_version` to add informative descriptions to the datasets.
- Handle missing descriptions by filling them with "Unknown diagnosis" or "Unknown procedure".
- Retain only necessary columns for further analysis:
    * diagnosis: `['subject_id', 'hadm_id', 'icd_version', 'diagnosis_description', 'icd_code']`
    * procedures: `['subject_id', 'hadm_id', 'icd_version', 'procedure_description', 'icd_code']`

#### Step 3: Compute Age at Events

- Filter patient data to retain only relevant i.e. `subject_id`, `gender`, `anchor_age`, `anchor_year`, `dod`.
- Merge patient demographic data with other datasets (admissions, ICU stays, diagnoses, procedures, prescriptions) to include age information.
- Convert relevant date fields (`admittime`, `intime`, and `starttime`) to `datetime` objects.
- Calculate patient age at event time using the auxiliary function `compute_age_at_event`, which adapts age calculations to each type of event's timestamp.
- Directly assign age for events (diagnoses and procedures) lacking precise timestamps.

#### Step 4: Generate Structured Clinical History

- Define the function `generate_patient_history` to compile clinical histories for individual patients.
- For each patient, include gender, death status, and age at death if applicable.
- Sort and iterate through patient hospitalizations, providing details such as:
  - Admission type (*emergency* or *normal*) and discharge status.
  - Diagnoses and procedures associated with each hospitalization.
  - Medications prescribed, detailing dosage and administration route.
  - Summary of ICU stays with length of stay, handling missing discharge dates (if missing we do not compute the icu stay length) to maintain informational integrity.

#### Step 5: Generate and Save Summaries

- Compile summaries for a selection of patients (restrained to 100 for testing) TODO: FOR ALL.
- Utilize a dictionary comprehension to generate and store histories in `summaries`.
- Save all clinical histories to a text file (`clinical_histories.txt`) for external review and use.


## Description of tabular-summary data preprocessing in preprocessiong_no_pyhealth.ipynb

This Python script processes clinical data from the MIMIC-IV database to generate a tabular dataset from textual patient history. It compiles key clinical features like admissions, diagnoses, procedures, ICU stays, and prescriptions into a structured format suitable for machine learning and analysis.

### Detailed Documentation

#### Step 1: Load Data

- Imports necessary libraries and loads datasets.
- Datasets include demographics (`patients`), admissions (`admissions`), ICU stays (`icustays`), medical records (`diagnoses`, `procedures`), and medication prescriptions (`prescriptions`).
- Descriptions for ICD codes are loaded to provide context to diagnoses and procedures.

#### Step 2: Map ICD Codes to Descriptions

- Merges diagnosis and procedure data with descriptions using a left merge on `icd_code` and `icd_version`.
- Ensures the inclusion of a human-readable description of diagnoses and procedures within patient records.

#### Step 3: Compute Age at Events & Mortality

- Filters patient data to retain core columns including `subject_id`, `gender`, `anchor_age`, etc.
- Merges demographic data with main tables to incorporate patient age and mortality status.
- Calculations:
  - `age_at_event` for hospital admissions and prescriptions.
  - Mortality indicators `death_flag` and `age_at_death`.

#### Step 4: Build Tabular Dataset

- Aggregates key features:
  - **ICU Stays**: Counts ICU admissions and calculates total stay days per hospitalization.
  - **Diagnoses**: Summarizes the number and descriptions of diagnoses per hospitalization.
  - **Procedures**: Counts and lists unique procedures performed.
  - **Medications**: Counts and lists prescribed drugs.
- Merges aggregated summaries with core admission features for a comprehensive dataset.
- Handles missing values by filling numerical with zeros and textual lists with "None".

#### Step 5: Save & Display Dataset

- Saves the compiled tabular dataset to a CSV file: `mimiciv_clinical_dataset_tabular.csv`.
- Prepares the dataset for further exploration or machine learning analysis, offering a concise view of patient histories in a structured, numerical format.


In [8]:
import pandas as pd
tabular_data = pd.read_csv("mimiciv_clinical_dataset_tabular.csv")
tabular_data.head(10)

,subject_id,hadm_id,age_at_event,gender,admission_type,discharge_location,death_flag,age_at_death,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list
0,10000032,22595853,52,F,URGENT,HOME,1,52.0,0.0,0.0,8.0,"['Portal hypertension', 'Other ascites', 'Cirr...",1.0,['Percutaneous abdominal drainage'],14.0,"['Furosemide', 'Ipratropium Bromide Neb', 'Pot..."
1,10000032,22841357,52,F,EW EMER.,HOME,1,52.0,0.0,0.0,8.0,['Unspecified viral hepatitis C with hepatic c...,1.0,['Percutaneous abdominal drainage'],15.0,"['Furosemide', 'Rifaximin', 'Sodium Chloride 0..."
2,10000032,25742920,52,F,EW EMER.,HOSPICE,1,52.0,0.0,0.0,10.0,['Chronic hepatitis C without mention of hepat...,1.0,['Percutaneous abdominal drainage'],28.0,"['Sodium Chloride 0.9% Flush', '0.9% Sodium C..."
3,10000032,29079034,52,F,EW EMER.,HOME,1,52.0,1.0,0.0,13.0,"['Other iatrogenic hypotension', 'Chronic hepa...",0.0,None,24.0,"['Bisacodyl', 'Senna', 'Calcium Carbonate', 'R..."
4,10000068,25022803,19,F,EU OBSERVATION,NaN,0,NaN,0.0,0.0,1.0,"['Alcohol abuse, unspecified']",1.0,['Other nonoperative respiratory measurements'],0.0,None
5,10000084,23052089,72,M,EW EMER.,HOME HEALTH CARE,1,73.0,0.0,0.0,6.0,"['Neurocognitive disorder with Lewy bodies', '...",0.0,None,13.0,"['Pramipexole', 'Pravastatin', 'rivastigmine',..."
6,10000084,29888819,72,M,EU OBSERVATION,NaN,1,73.0,0.0,0.0,6.0,"['Altered mental status, unspecified', ""Parkin...",0.0,None,0.0,None
7,10000108,27250926,25,M,EU OBSERVATION,NaN,0,NaN,0.0,0.0,2.0,['Cellulitis and abscess of oral soft tissues'...,0.0,None,0.0,None
8,10000117,22927623,55,F,EU OBSERVATION,NaN,0,NaN,0.0,0.0,9.0,"['Dysphagia, unspecified', 'Other specified sy...",0.0,None,2.0,"['Heparin', 'Sodium Chloride 0.9% Flush']"
9,10000117,27988844,57,F,OBSERVATION ADMIT,HOME HEALTH CARE,0,NaN,0.0,0.0,13.0,['Unspecified intracapsular fracture of left f...,1.0,['Reposition Left Upper Femur with Internal Fi...,27.0,"['Iso-Osmotic Dextrose', 'CeFAZolin', 'Vitamin..."


### Considerations:
In this setting we are still missing some key feature with a lot of potential:
* How to use the `labevents` dataset of the `hosp` module?
* `pharmacy` dataset, should also be included? In my opinon no (all the 'important' info are contained already in the prescriptions), but say why.
* Other ICU modules, like `chartevents` (Contains the majority of information documented in the ICU) and `datetimeevents`.
* Is missing the info related to the days between each visit for each patients. This information could be useful for considering the readmission event (we could to model this rather than mortality)
* Is missing a variable which says if the patients died in the visit or not (i.e. lack of mortality definition). To this end the variable `death_flag` is not very clear on what represents (yes, if the patients died, but not specified exactly when - at least in term of visit)

Then, a great open question remains to understand how the data in this format could been use to train standard ML models (LG, RF, Boosting, etc.), profiding example on how other studies performed this (like https://clinicalbench.github.io).

Now, for the latter two open points we created an updated version of that dataset, called `mimiciv_clinical_dataset_tabular_death_visit.csv`. Let us see here a small fraction of the dataset:

In [4]:
tabular_data_refined = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit.csv")
tabular_data_refined.head(5)

,subject_id,hadm_id,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,age_at_death,died_during_visit,days_from_last_visit_to_death,days_until_next_visit,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list
0,10000032,22595853,52,F,URGENT,EMERGENCY,HOME,1,52.0,0,32.0,50.0,0.0,0.0,8.0,"['Portal hypertension', 'Other ascites', 'Cirr...",1.0,['Percutaneous abdominal drainage'],14.0,"['Furosemide', 'Ipratropium Bromide Neb', 'Pot..."
1,10000032,22841357,52,F,EW EMER.,EMERGENCY,HOME,1,52.0,0,32.0,25.0,0.0,0.0,8.0,['Unspecified viral hepatitis C with hepatic c...,1.0,['Percutaneous abdominal drainage'],15.0,"['Furosemide', 'Rifaximin', 'Sodium Chloride 0..."
2,10000032,29079034,52,F,EW EMER.,EMERGENCY,HOME,1,52.0,0,32.0,11.0,1.0,0.0,13.0,"['Other iatrogenic hypotension', 'Chronic hepa...",0.0,None,24.0,"['Bisacodyl', 'Senna', 'Calcium Carbonate', 'R..."
3,10000032,25742920,52,F,EW EMER.,EMERGENCY,HOSPICE,1,52.0,0,32.0,-1.0,0.0,0.0,10.0,['Chronic hepatitis C without mention of hepat...,1.0,['Percutaneous abdominal drainage'],28.0,"['Sodium Chloride 0.9% Flush', '0.9% Sodium C..."
4,10000068,25022803,19,F,EU OBSERVATION,NORMAL,NaN,0,NaN,0,NaN,-1.0,0.0,0.0,1.0,"['Alcohol abuse, unspecified']",1.0,['Other nonoperative respiratory measurements'],0.0,None


As we can see here we have more useful variables than in the previous version of the dataset. In particular, we have the following:
* `admission_category`: we simplify the 9-categories `admission_type` variable in 2-categories variable, where we grouped toghether emergency-values and we set as "normal" the others.
* `died_during_visit`: this variable tells us whether the patients died during the hospitalization or not. If not, we keep track on how many days after the last hospitalization the patient died.
* `days_from_last_visit_to_death`: this variable tells us (if the patient died outside an hospitalization) after how many days from last visit the patient died. **NOTE: If the patient died after one year from the last visit, then it's not reported. That's from how MIMIC-IV data is built.**
* `days_until_next_visit`: this variable tells us how many days we have from one visit to the next one, for each patient. If the patient has only one visit, we then have -1 as a value.

Now, we should focus on one aspect, let us see how many unique individuals we have in this dataset:

In [5]:
len(tabular_data_refined['subject_id'].unique())

223452

In the documentation of MIMIC, however, we have:
_As of MIMIC-IV v3.0 there are 364,627 unique patients, of whom 223,452 had at least one hospitalization (i.e. at least one record in the admissions table). The remaining 141,175 patients were only seen in the emergency department, which can be verified using the transfers table._

So, we have 141175 patients who are in the ICU department but not in the hospitalization. We should investigate further how we should consider these patients for example: 
* do they have as well some dod information?
* is it possible to have different ICU admission for the same individual?

In [7]:
import pandas as pd

# Load necessary MIMIC-IV tables
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")      # Contains subject_id, dod
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")  # Contains subject_id, hadm_id
transfers = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/transfers.csv.gz", compression="gzip")    # Contains subject_id, eventtype
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")       # Contains subject_id, hadm_id, stay_id

# 1. Identify ED-only patients
# Patients in transfers but NOT in admissions
ed_only_patients = set(transfers["subject_id"]) - set(admissions["subject_id"])
ed_only_df = transfers[transfers["subject_id"].isin(ed_only_patients)]

# 2. Check how many ED-only patients have a recorded date of death
ed_only_with_dod = patients[patients["subject_id"].isin(ed_only_patients) & patients["dod"].notna()]
print(f"Number of ED-only patients: {len(ed_only_patients)}")
print(f"ED-only patients with recorded DOD: {len(ed_only_with_dod)}")

# 3. Check for multiple ICU stays per patient
icustay_counts = icustays.groupby("subject_id")["stay_id"].count().reset_index()
multiple_icu_stays = icustay_counts[icustay_counts["stay_id"] > 1]
print(f"Number of patients with multiple ICU stays: {len(multiple_icu_stays)}")

# Show ICU stays for ED-only patients
ed_only_icu_stays = icustays[icustays["subject_id"].isin(ed_only_patients)]
print(f"Number of ED-only patients with ICU stays: {len(ed_only_icu_stays['subject_id'].unique())}")

# Display results in a DataFrame
results_df = pd.DataFrame({
    "Total ED-Only Patients": [len(ed_only_patients)],
    "ED-Only Patients with DOD": [len(ed_only_with_dod)],
    "Patients with Multiple ICU Stays": [len(multiple_icu_stays)],
    "ED-Only Patients with ICU Stays": [len(ed_only_icu_stays['subject_id'].unique())]
})

# Display DataFrame
print("\nSummary of MIMIC-IV Analysis:")
display(results_df)


Number of ED-only patients: 141175
ED-only patients with recorded DOD: 1419
Number of patients with multiple ICU stays: 16242
Number of ED-only patients with ICU stays: 0

Summary of MIMIC-IV Analysis:


,Total ED-Only Patients,ED-Only Patients with DOD,Patients with Multiple ICU Stays,ED-Only Patients with ICU Stays
0,141175,1419,16242,0


It could be that  these patients does have a record in the `patients` file but not in the `admissions` one (or in other file).

In [9]:
# 2. Check if these patients exist in patients.csv
ed_only_in_patients = patients[patients["subject_id"].isin(ed_only_patients)]

# 3. Count how many ED-only patients exist in patients.csv
num_ed_only_in_patients = len(ed_only_in_patients)
num_ed_only_missing = len(ed_only_patients) - num_ed_only_in_patients

print(f"Total ED-only patients: {len(ed_only_patients)}")
print(f"ED-only patients found in patients.csv: {num_ed_only_in_patients}")
print(f"ED-only patients missing from patients.csv: {num_ed_only_missing}")

# Display results in a DataFrame
results_df = pd.DataFrame({
    "Total ED-Only Patients": [len(ed_only_patients)],
    "ED-Only Patients in patients.csv": [num_ed_only_in_patients],
    "ED-Only Patients Missing in patients.csv": [num_ed_only_missing]
})

# Show DataFrame
display(results_df)

Total ED-only patients: 141175
ED-only patients found in patients.csv: 141175
ED-only patients missing from patients.csv: 0


,Total ED-Only Patients,ED-Only Patients in patients.csv,ED-Only Patients Missing in patients.csv
0,141175,141175,0


Okay, so, we have all the patients information in the patients.csv file (where we have also all the dod info).
There are patients registered in the `patients.csv` file but that are missing from the hosp module or icu module. For these patients we could investigate whom they are, but we can do as well nothing since they could be related to other things. However we should carefully check whether there are some patients in the **ED** unit that are not in the ICU module.

I downloaded this dataset in `edstays.csv` from https://physionet.org/content/mimic-iv-ed/2.2/ed/#files-panel and I placed in the `data/` folder so it's in the `.gitignore` file.

We can now proceed with this check.

In [ ]:
import pandas as pd

# Load necessary MIMIC-IV tables
edstays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/edstays.csv")  # Contains ED patients
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")  # Contains hospital admissions
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")  # Contains ICU stays

# 1. Identify all unique ED patients
ed_patients = set(edstays["subject_id"])

# 2. Find ED patients who were NEVER admitted to the hospital
ed_not_admitted = ed_patients - set(admissions["subject_id"])

# 3. Find ED patients who were NEVER admitted to the ICU
ed_not_in_icu = ed_patients - set(icustays["subject_id"])

# 4. Count how many ED patients were in neither ICU nor Hospital
#ed_only_patients = ed_not_admitted & ed_not_in_icu
ed_only_patients = ed_not_admitted.intersection(ed_not_in_icu)

# Display results
print(f"Total patients in ED module: {len(ed_patients)}")
print(f"ED patients NOT admitted to hospital: {len(ed_not_admitted)}")
print(f"ED patients NOT admitted to ICU: {len(ed_not_in_icu)}")
print(f"ED patients in neither Hospital nor ICU (ED-only): {len(ed_only_patients)}")

# Create a DataFrame for results
results_df = pd.DataFrame({
    "Total ED Patients": [len(ed_patients)],
    "ED Patients NOT Admitted to Hospital": [len(ed_not_admitted)],
    "ED Patients NOT Admitted to ICU": [len(ed_not_in_icu)],
    "ED-Only Patients (Neither Hospital nor ICU)": [len(ed_only_patients)]
})

# Display DataFrame
display(results_df)


Total patients in ED module: 205504
ED patients NOT admitted to hospital: 79409
ED patients NOT admitted to ICU: 171811
ED patients in neither Hospital nor ICU (ED-only): 79409


,Total ED Patients,ED Patients NOT Admitted to Hospital,ED Patients NOT Admitted to ICU,ED-Only Patients (Neither Hospital nor ICU)
0,205504,79409,171811,79409


In [13]:
# 2. Find ED patients who were admitted to the hospital
ed_admitted_to_hospital = ed_patients.intersection(set(admissions["subject_id"]))

# 3. Find ED patients who were admitted to the ICU
ed_admitted_to_icu = ed_patients.intersection(set(icustays["subject_id"]))

# 4. Find ED patients who were admitted to the hospital but NOT the ICU
ed_hospital_not_icu = ed_admitted_to_hospital - ed_admitted_to_icu

# Display results
print(f"Total ED Patients: {len(ed_patients)}")
print(f"ED Patients Admitted to Hospital: {len(ed_admitted_to_hospital)}")
print(f"ED Patients Admitted to ICU: {len(ed_admitted_to_icu)}")
print(f"ED Patients Admitted to Hospital but NOT ICU: {len(ed_hospital_not_icu)}")

# Create DataFrame for results
results_df = pd.DataFrame({
    "Total ED Patients": [len(ed_patients)],
    "ED Patients Admitted to Hospital": [len(ed_admitted_to_hospital)],
    "ED Patients Admitted to ICU": [len(ed_admitted_to_icu)],
    "ED Patients Admitted to Hospital but NOT ICU": [len(ed_hospital_not_icu)]
})

# Show DataFrame
display(results_df)

Total ED Patients: 205504
ED Patients Admitted to Hospital: 126095
ED Patients Admitted to ICU: 33693
ED Patients Admitted to Hospital but NOT ICU: 92402


,Total ED Patients,ED Patients Admitted to Hospital,ED Patients Admitted to ICU,ED Patients Admitted to Hospital but NOT ICU
0,205504,126095,33693,92402


In [14]:
# great. If we substract from the 171811 patients not admitted to ICU the 92402 (ED Patients Admitted to Hospital but NOT ICU) we get:
len(ed_not_in_icu) - len(ed_hospital_not_icu)

79409

Which is exactly the number of ED-Only Patients (Neither Hospital nor ICU).

Now, of these 79409 patients, are there any emergency?

In [15]:
ed_only_patients

{17301508,
 16252935,
 12058642,
 19136531,
 19922962,
 14417948,
 10747933,
 19398689,
 19922978,
 13369377,
 18087973,
 15990823,
 11272232,
 14417960,
 17563689,
 16252973,
 13107247,
 18350131,
 15466553,
 10485818,
 19660857,
 13893693,
 19136574,
 18874431,
 10223689,
 14417994,
 18874461,
 11534429,
 12320862,
 11272290,
 12058728,
 13107306,
 18088045,
 11534447,
 13369457,
 13107314,
 18088049,
 11272313,
 18350202,
 18874492,
 19660925,
 17825919,
 14680193,
 19136646,
 15466635,
 17825939,
 13107350,
 12583064,
 17039513,
 19923102,
 14155935,
 12583076,
 16253095,
 16777383,
 17301671,
 12320940,
 13369516,
 17563823,
 16777394,
 14155959,
 17301692,
 13893825,
 15466701,
 10223823,
 19136720,
 10748130,
 11534563,
 13107427,
 15728874,
 12845291,
 13893867,
 13369581,
 16777454,
 15728880,
 10223856,
 10748153,
 15466746,
 18088202,
 14418189,
 15991061,
 19661080,
 11272474,
 12845339,
 15728927,
 15204642,
 18612515,
 16777508,
 18088228,
 19136805,
 15204651,
 10223915,